In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pickle

from config import (
    DATA_DIR,
    P,
    N_SUBJECTS,
    SPARSITY,
    SNR
)

from data_utils import (
    generate_simulation_data,
    load_fmri_data
)

from cross_validation import (
    run_cv_over_lambdas
)

from plot_utils import (
    plot_X1_X2_heatmaps,
    plot_train_test_grid_all_lambdas,
    plot_r2_over_lambda,
    plot_fold_r2_by_lambda
)

In [ ]:
X1_sim, X2_sim, y_sim, beta1_true, beta2_true = (
    generate_simulation_data(
        data_dir=DATA_DIR,
        p=P,
        n=N_SUBJECTS,
        sparsity=SPARSITY,
        snr=SNR,
        rs_file="z_fc.npy",
        emo_file="z_sc.npy",
    )
)



X1, X2, y, age = load_fmri_data(
    data_dir=DATA_DIR,
    p=P,
    n_subjects=N_SUBJECTS,

    # Change these for PMAT / WRAT / sex-specific analyses
    rs_file="z_fc.npy",
    emo_file="z_sc.npy",
    y_file="y.npy",
    age_file="age.npy"
)

X2 = np.log1p(X2)

In [ ]:
y_sim_c = (
    np.asarray(y_sim)
    -
    np.mean(y_sim)
)

age_c = (
    np.asarray(age)
    -
    np.mean(age)
)

In [ ]:
plt.figure(
    figsize=(7, 4)
)

plt.hist(
    age_c,
    bins=30,
    alpha=0.5,
    density=True,
    label="Real y"
)

plt.hist(
    y_sim_c,
    bins=30,
    alpha=0.5,
    density=True,
    label="Simulated y"
)

plt.xlabel(
    "Centered outcome"
)

plt.ylabel(
    "Density"
)

plt.legend()

plt.title(
    "Real and simulated outcomes"
)

plt.show()

In [ ]:
plot_X1_X2_heatmaps(
    X1,
    X2,
    subject=20
)

In [ ]:
real_results, real_summary = (
    run_cv_over_lambdas(
        X1=X1,
        X2=X2,
        y=age_c,

        # Age is residualized INSIDE each CV fold
        age=None,

        seed=2026
    )
)

print(
    "\nReal-data CV results"
)

print(
    real_summary
)

In [ ]:
real_summary

In [ ]:
plot_train_test_grid_all_lambdas(
    real_results,
    save_path="figures/train_test_grid_all_lambdas_real_data2.pdf"
)

In [ ]:
plot_fold_r2_by_lambda(
    real_results
)

In [ ]:
from bilinear_lasso import bilinear_lasso
import jax.numpy as jnp
STEP_SIZE = 1e-6
TOL = 1e-5

X1_c = X1 - X1.mean(axis=2, keepdims=True)
X2_c = X2 - X2.mean(axis=2, keepdims=True)
final_model = bilinear_lasso(X1_c, X2_c, age_c, 4, 4)
final_model.initialize_beta(jnp.zeros((P, 1)), jnp.zeros((P, 1)))
final_model.fit(num_candidates=2, max_iter=6000, step_size=STEP_SIZE, tol=TOL, disturbance=2)


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
# axs[0].bar(range(len(model.beta1)), np.array(model.beta1).flatten())
axs[0].hist(np.array(final_model.beta1).flatten())
axs[0].set_title('Estimated beta1')
axs[0].set_xlabel('Node index')
# axs[1].bar(range(len(model.beta2)), np.array(model.beta2).flatten())
axs[1].hist(np.array(final_model.beta2).flatten())
axs[1].set_title('Estimated beta2')
axs[1].set_xlabel('Node index')
plt.tight_layout()
plt.show()

In [ ]:
y_true = np.array(age_c).ravel()
y_hat = (np.einsum('ij,ijk,ji->k', final_model.beta1, np.array(X1_c), final_model.beta1) +
         np.einsum('ij,ijk,ji->k', final_model.beta2, np.array(X2_c), final_model.beta2))
residuals = y_true - y_hat

corr = np.corrcoef(y_true, y_hat)[0, 1]
r2   = 1 - np.var(residuals) / np.var(y_true)

beta1_traj = np.array(final_model.trajectory[final_model.sel_idx]['beta1'])  # (num_steps, p, 1)
beta2_traj = np.array(final_model.trajectory[final_model.sel_idx]['beta2'])  # (num_steps, p, 1)


plt.figure(figsize=(20, 4))

# Panel 1: First 50 subjects, line+dot style
plt.subplot(1, 5, 1)
plt.plot(y_true, 'o', markersize=4, linewidth=1, label='True y')
plt.plot(y_hat,  'x', markersize=4, linewidth=1, label='Predicted y')
plt.title(f'True vs Predicted y (first 50)\nr={corr:.3f}, R²={r2:.3f}')
plt.xlabel('Subject index')
plt.ylabel('y')
plt.legend(fontsize=7)
plt.grid(True)

# Panel 2: Sorted by true y — predicted first (behind), true on top
plt.subplot(1, 5, 2)
idx = np.argsort(y_true)
plt.plot(y_hat[idx],  'x', markersize=2, linewidth=0.8, label='Predicted y')
plt.plot(y_true[idx], 'o',  markersize=2, linewidth=0.8, label='True y')
plt.title('True vs Predicted y\n(sorted by true y)')
plt.xlabel('Subject (sorted)')
plt.ylabel('y')
plt.legend(fontsize=7)
plt.grid(True)

# Panel 3: Residual plot
plt.subplot(1, 5, 3)
plt.scatter(y_true, y_hat, s=10, alpha=0.5)
min_val = min(y_true.min(), y_hat.min())
max_val = max(y_true.max(), y_hat.max())
plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='Perfect Fit')
plt.title('Actual vs. Predicted')
plt.xlabel('True Values')
plt.ylabel('Predicted Values')
plt.grid(True)

# Panel 4: Beta1 trajectories (first 5 components)
plt.subplot(1, 5, 4)
for i in range(beta1_traj.shape[1]):
    plt.plot(beta1_traj[:, i, 0], label=f'beta1[{i}]')
plt.title('Beta1 Trajectories')
plt.xlabel('Iteration')
plt.ylabel('Beta1 value')
plt.grid(True)

# Panel 5: Beta2 trajectories (first 5 components)
plt.subplot(1, 5, 5)
for i in range(beta2_traj.shape[1]):
    plt.plot(beta2_traj[:, i], label=f'beta2[{i}]')
plt.title('Beta2 Trajectories')
plt.xlabel('Iteration')
plt.ylabel('Beta2 value')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable
# Use the same color scale
beta1 = np.asarray(final_model.beta1)
beta2 = np.asarray(final_model.beta2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

axes[0].plot(beta1)
axes[0].axhline(0, linestyle="--")
axes[0].set_title(r"$\beta_1$")
axes[0].set_xlabel("ROI")
axes[0].set_ylabel("Coefficient")

axes[1].plot(beta2)
axes[1].axhline(0, linestyle="--")
axes[1].set_title(r"$\beta_2$")
axes[1].set_xlabel("ROI")

plt.tight_layout()
plt.show()